In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Run once. You can install via requirements.txt:
# %pip install -q -r requirements.txt

# Or run the direct installs below:
%pip install -q \
    langchain langchain-community langchain-core \
    chromadb \
    pypdf python-docx beautifulsoup4 requests \
    tiktoken \
    langchain-openai langchain-anthropic langchain-ollama sentence-transformers

print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
# Edit this cell to switch providers. Everything else adapts automatically.

from rag import RAGConfig

# ── Provider & Model Selection ───────────────────────────────────────────────
config = RAGConfig(
    llm_provider="openai",       # "openai" | "anthropic" | "ollama"
    embedding_provider="openai", # "openai" | "huggingface" | "ollama"
    
    # Optional custom models (leave empty to use default mappings)
    llm_model="",
    embedding_model="",
    
    # ChromaDB & Chunking Settings
    chroma_persist_dir="./chroma_db",
    collection_name="rag_docs",
    chunk_size=1000,
    chunk_overlap=150,
    top_k=5,
)

print(f"LLM:        {config.llm_provider} / {config.llm_model}")
print(f"Embeddings: {config.embedding_provider} / {config.embedding_model}")
print(f"Vector DB:  ChromaDB → {config.chroma_persist_dir}/{config.collection_name}")

In [ ]:
# ── Cell 3: Load & Ingest Documents ──────────────────────────────────────────
from rag import RAGPipeline

# ✏️  Edit the lists below to point at YOUR data sources.
# Leave any list empty [] if you don't have that type.

# Paths to local PDF files
PDF_PATHS = [
    # "./docs/report.pdf",
    # "./docs/manual.pdf",
]

# Paths to local .docx files
DOCX_PATHS = [
    # "./docs/notes.docx",
]

# Local .txt or .md files
TEXT_PATHS = [
    # "./docs/readme.md",
    # "./docs/notes.txt",
]

# Web URLs to scrape
WEB_URLS = [
    # "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
]

# ── Initialize Pipeline and Run Ingest ────────────────────────────────────────
pipeline = RAGPipeline(config)

if any([PDF_PATHS, DOCX_PATHS, TEXT_PATHS, WEB_URLS]):
    chunks = pipeline.ingest(
        pdf_paths=PDF_PATHS,
        docx_paths=DOCX_PATHS,
        text_paths=TEXT_PATHS,
        web_urls=WEB_URLS,
    )
    print(f"\n✅ Ingested {len(chunks)} chunks into ChromaDB.")
else:
    count = pipeline.vectorstore._collection.count()
    if count > 0:
        print(f"ℹ️  Using existing index with {count} chunks (no new documents to add).")
    else:
        print("⚠️  Vector store is empty. Add paths/URLs above to ingest documents.")

In [ ]:
# ── Cell 4: Query the Pipeline ────────────────────────────────────────────────
# Replace this with questions relevant to YOUR documents.

answer = pipeline.ask("What is this document about?", show_sources=True)

In [2]:
# ── Cell 5: Batch Evaluation ─────────────────────────────────────────────────
# Define question + expected keywords to do a quick sanity check.

eval_cases = [
    # {"question": "What is the main topic?",   "must_contain": ["topic_keyword"]},
    # {"question": "Who are the authors?",       "must_contain": ["author_name"]},
]

pipeline.evaluate(eval_cases)

SyntaxError: invalid decimal literal (3281415745.py, line 12)

---

## Tuning Guide

| Problem | What to try |
|---|---|
| Answers miss key facts | Increase `TOP_K` or reduce `CHUNK_SIZE` so chunks are more focused |
| Answers are too fragmented | Increase `CHUNK_SIZE` and `CHUNK_OVERLAP` |
| LLM hallucinates | Lower temperature (already 0); tighten the prompt |
| Slow embedding | Switch `EMBEDDING_PROVIDER` to `"huggingface"` (local, no API cost) |
| Retrieval returns irrelevant chunks | Switch `search_type` from `"mmr"` to `"similarity"` |
| Too expensive | Use `gpt-4o-mini` or switch to `ollama` for fully local inference |

## Next steps

- **Hybrid search** — combine BM25 keyword search with dense vector retrieval (`EnsembleRetriever`)
- **Re-ranking** — add a cross-encoder reranker (e.g. `FlashrankRerank`) after initial retrieval  
- **Metadata filtering** — filter by document type, date, or author at query time  
- **Multi-query retrieval** — generate 3–5 query variants and union the results  
- **Evaluation framework** — use RAGAS or LangChain Evals for faithfulness/relevance scoring